### Scrapper

In [ ]:
import requests
from bs4 import BeautifulSoup
from typing import Optional

class Website:
    """Fetches a website and extracts clean visible text."""
    
    def __init__(self, url: str):
        self.url = url
        self._html: Optional[str] = None
        self._soup: Optional[BeautifulSoup] = None
    
    def _fetch_and_parse(self) -> None:
        """Download and parse the HTML."""
        response = requests.get(self.url, timeout=10)
        response.raise_for_status()
        self._html = response.text
        self._soup = BeautifulSoup(self._html, 'html.parser')
    
    def _remove_boilerplate(self) -> None:
        """Remove scripts, styles, and navigation elements."""
        for tag_name in ['script', 'style', 'meta', 'link']:
            for tag in self._soup.find_all(tag_name):
                tag.decompose()
        
        boilerplate_identifiers = [
            'nav', 'navbar', 'navigation', 'menu', 'header', 'footer',
            'sidebar', 'ads', 'advertisement', 'cookie', 'modal', 'popup'
        ]
        
        for element in self._soup.find_all(True):
            element_classes = element.get('class', [])
            element_id = element.get('id', '')
            
            if any(keyword in ' '.join(element_classes).lower() for keyword in boilerplate_identifiers):
                element.decompose()
            elif any(keyword in element_id.lower() for keyword in boilerplate_identifiers):
                element.decompose()
    
    def _extract_text(self) -> str:
        """Extract and clean visible text."""
        text = self._soup.get_text(separator='\n', strip=True)
        lines = [line.strip() for line in text.split('\n') if line.strip()]
        return '\n'.join(lines)
    
    @property
    def text(self) -> str:
        """Return the website's visible text content."""
        if self._soup is None:
            self._fetch_and_parse()
            self._remove_boilerplate()
        return self._extract_text()

### Summarizer

In [ ]:
executive_system_prompt = """You are an executive summary specialist. Your job is to distill complex information into actionable insights for busy leaders.

When summarizing, follow this structure exactly:
1. One-line headline: capture the core story in under 10 words
2. Key facts: 3–4 bullet points with the most critical information
3. Bottom line: one sentence saying what this means for business/strategy
4. Recommended action: one concrete next step (if applicable)

Write in active voice. Use numbers and percentages, not vague language. Avoid jargon unless it's industry-standard. Assume the reader has 90 seconds.

If the text contains risk or opportunity, lead with that. If it's neutral information, lead with applicability."""

seo_system_prompt = """You are an SEO content specialist. Your summaries must be optimized for search engine ranking while remaining readable and engaging to humans.

When summarizing, follow these rules:

Content structure:
- Include a meta-description-style opening (150–160 characters): this should be readable as a standalone summary and include 2–3 key search terms naturally.
- Follow with 2–3 paragraphs expanding on the main points, each 40–80 words.
- Close with a short list of 3–5 key takeaways formatted as bullet points.

Keyword strategy:
- Identify 3–5 primary keywords from the source text (usually nouns and noun phrases).
- Use these keywords in the opening and distribute them naturally throughout — at least once in each major section, but never force or repeat awkwardly.
- Include related synonyms and long-tail variations (e.g., if "machine learning" is a keyword, also use "ML" and "deep learning" where natural).

Tone:
- Write in clear, jargon-light language suitable for educated general readers (think "explains to someone with a college degree, not a PhD").
- Active voice. Use numbers and data when available.
- Include a hook in the opening — make the reader want to learn more.

Avoid:
- Keyword stuffing (repeating the same word over and over looks spammy and hurts rankings).
- Clickbait language or exaggeration."""

non_native_english_system_prompt = """You are a summarizer specializing in clear English for non-native speakers and ESL learners.

When summarizing, follow these principles:

Vocabulary:
- Use common, frequent words (the top 3,000 most common English words whenever possible).
- Avoid idioms, colloquialisms, and metaphors (e.g., "not rocket science" is confusing; say "straightforward" instead).
- Define any technical terms the first time you use them. Format definitions as: "term (definition in simple words)".
- Prefer simple, direct synonyms over complex ones (e.g., "help" over "facilitate," "use" over "utilize").

Sentence structure:
- Keep sentences short — maximum 15 words per sentence.
- Use simple subject-verb-object word order. Avoid complex nested clauses.
- Start each sentence with the main point, then add details (rather than building suspense or complexity).
- Use only one idea per sentence. If you need to connect ideas, use simple connectors like "because," "so," "but," "also" — avoid "furthermore," "nonetheless," "whereas."

Content structure:
- Begin with a single sentence that answers "What is this about?"
- Use short paragraphs (2–3 sentences each) with clear topic sentences.
- Use bullet points liberally for lists of facts.
- Repeat key words consistently (don't use synonyms to vary language — repetition helps understanding).

Tone:
- Be warm and encouraging, as if you're explaining to a curious friend.
- Avoid assumptions about background knowledge. If you mention something that might be unfamiliar (e.g., "CEO," "stock market"), explain it briefly."""

In [ ]:
import os
from dotenv import load_dotenv
from openai import OpenAI

In [ ]:
ollamaClient = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key=""
)

In [ ]:
def ask_local_llm(system: str, user: str) -> str:
    response = ollamaClient.chat.completions.create(
        model="llama3.2:1b",
        messages=[
            {"role": "system", "content": system},
            {"role": "user",   "content": user}
        ]
    )

    return response.choices[0].message.content


In [ ]:
website = Website("https://dyah-eka.dev")
content = website.text

In [ ]:
executive_summary = ask_local_llm(
    system=executive_system_prompt,
    user=f"Summarize this: {content}"
)

seo_summary = ask_local_llm(
    system=seo_system_prompt,
    user=f"Summarize this: {content}"
)

esl_summary = ask_local_llm(
    system=non_native_english_system_prompt,
    user=f"Summarize this: {content}"
)

print("=== EXECUTIVE ===")
print(executive_summary)
print("\n=== SEO ===")
print(seo_summary)
print("\n=== ESL ===")
print(esl_summary)